# 3. Every-Visit Monte Carlo
### Updates Using Every Visit to a State

In **Every-Visit Monte Carlo**, the return $G_t$ is recorded and averaged for **every occurrence** of state $s$ in an episode, even if the state is visited multiple times.

#### First-Visit vs. Every-Visit Comparison:
| Property | First-Visit MC | Every-Visit MC |
|---|---|---|
| **Visits counted** | Only first visit in an episode | Every occurrence in an episode |
| **Bias** | Strictly unbiased ($E[G_t] = V(s)$) | Slightly biased in small samples, asymptotically unbiased |
| **Variance / MSE** | Higher variance initially | Often lower Mean Squared Error (MSE) initially |
| **Complexity** | Requires tracking earlier states | Simpler to implement |


In [ ]:
# Step 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


In [ ]:
# Step 2: Define a Simple Looping MDP Environment
# States: 0, 1, 2, with self-loops so multiple visits occur within an episode
class LoopingEnv:
    def __init__(self):
        self.num_states = 3
        
    def reset(self):
        self.state = 0
        return self.state
        
    def step(self):
        # State transitions with probabilistic loop
        prob = np.random.rand()
        if self.state == 0:
            self.state = 0 if prob < 0.4 else 1
            reward = 0.0
            done = False
        elif self.state == 1:
            self.state = 1 if prob < 0.4 else 2
            reward = 0.0
            done = False
        else: # Terminal state 2
            reward = 1.0
            done = True
        return self.state, reward, done

loop_env = LoopingEnv()
print("Looping environment initialized.")


In [ ]:
# Step 3: Implement Both First-Visit and Every-Visit MC
def run_mc_comparison(env, num_episodes=5000, gamma=0.9):
    V_first = defaultdict(float)
    N_first = defaultdict(int)
    
    V_every = defaultdict(float)
    N_every = defaultdict(int)
    
    for _ in range(num_episodes):
        # Generate episode
        episode = []
        s = env.reset()
        while True:
            next_s, r, done = env.step()
            episode.append((s, r))
            s = next_s
            if done:
                break
                
        # 1. First-Visit MC
        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = gamma * G + r_t
            if s_t not in [step[0] for step in episode[:t]]:
                N_first[s_t] += 1
                V_first[s_t] += (G - V_first[s_t]) / N_first[s_t]
                
        # 2. Every-Visit MC
        G = 0.0
        for t in reversed(range(len(episode))):
            s_t, r_t = episode[t]
            G = gamma * G + r_t
            N_every[s_t] += 1
            V_every[s_t] += (G - V_every[s_t]) / N_every[s_t]
            
    return V_first, V_every

v_f, v_e = run_mc_comparison(loop_env, num_episodes=10000)
for s in [0, 1]:
    print(f"State {s} -> First-Visit V: {v_f[s]:.4f} | Every-Visit V: {v_e[s]:.4f}")


In [ ]:
# Step 4: Comparison Plot
states = [0, 1]
x = np.arange(len(states))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, [v_f[s] for s in states], width, label='First-Visit MC', color='royalblue')
ax.bar(x + width/2, [v_e[s] for s in states], width, label='Every-Visit MC', color='coral')

ax.set_ylabel('Estimated State-Value V(s)')
ax.set_title('First-Visit vs Every-Visit Monte Carlo Estimates')
ax.set_xticks(x)
ax.set_xticklabels([f'State {s}' for s in states])
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


### Summary
- In episodes where states recur frequently, Every-Visit MC extracts more learning updates per episode.
- Both estimators asymptotically converge to the true value function $V^\pi(s)$.
